In [5]:
import pandas as pd
import random
import os
import numpy as np

import csv
import time
from mlx_lm import load, generate

from huggingface_hub import InferenceClient
from huggingface_hub.utils import HfHubHTTPError

os.environ["TOKENIZERS_PARALLELISM"] = "false"

### Load GTDB genome info

In [6]:
base_dir = '/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/code/'

In [7]:
gtdb = pd.read_csv(f'{base_dir}/GTDB/MIDAS_GTDB_metadata_by_species.tsv', sep='\t')
gtdb['species'] = gtdb['GTDB species'].str.split('s__', expand=True)[1]
gtdb_species_counts = gtdb.set_index('species')['No. clustered genomes']

In [8]:
midas_genomes = pd.read_csv('/Users/cdubin/Library/CloudStorage/Box-Box/PTB_shotgun/MIDAS/GTDB_db/GTDB_genomes_in_MIDAS.tsv', sep='\t')
midas_genomes.shape

(258405, 4)

In [9]:
midas_genome_list = midas_genomes['genome'].unique().tolist()
len(midas_genome_list)

258405

## Get genome metadata from NCBI

In [14]:

pd.DataFrame(midas_genome_list).drop_duplicates().to_csv('accession_list.txt', header=False, index=False)
!wc -l accession_list.txt

  258405 accession_list.txt



### Get additional annotation info from NCBI

        datasets summary genome accession --inputfile accession_list.txt --as-json-lines |  
        dataformat tsv genome --fields accession,assminfo-bioproject-lineage-title,assminfo-biosample-bioproject-title,assminfo-biosample-project-name,assminfo-biosample-description-title,assminfo-biosample-breed,assminfo-biosample-host,assminfo-biosample-ecotype,assminfo-biosample-cultivar,assminfo-biosample-host-disease,assminfo-biosample-isolation-source,assminfo-biosample-source-type,assminfo-biosample-tissue,assminfo-biosample-models --force > NCBI_genome_info.tsv


Note: There are some accessions with duplicate info because it was reused in multiple studies; e.g. a genome that was originally published and then reuploaded because it's part of HMP

In [12]:
ncbi_info = pd.read_csv('NCBI_genome_info.tsv', sep='\t')
ncbi_info['Assembly BioSample Source type'] = ncbi_info['Assembly BioSample Source type'].str.lower()
ncbi_info['Assembly BioSample Host'] = ncbi_info['Assembly BioSample Host'].str.lower()
ncbi_info['Assembly BioSample Isolation source'] = ncbi_info['Assembly BioSample Isolation source'].str.lower()

ncbi_info = ncbi_info.drop_duplicates()

/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_28380/401415038.py:1: DtypeWarning: Columns (2,5,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  ncbi_info = pd.read_csv('NCBI_genome_info.tsv', sep='\t')


In [13]:
ncbi_info['Assembly Accession'].nunique()

258409

### Subset to just human-derived samples

In [14]:
import re

def is_human(row):
    search_list = r'(homo sapiens|human)'  # regex pattern

    if re.search(search_list, str(row['Assembly BioSample Host']), flags=re.IGNORECASE):
        return True
    
    non_human_hosts = r'(bos taurus|pig|sus scrofa|sus scrofa scrofa|chicken|fly|gallus gallus|equus ferus caballus|frog|dog)' 
    if re.search(non_human_hosts, str(row['Assembly BioSample Host']), flags=re.IGNORECASE):
        return False

    if re.search(search_list, str(row['Assembly BioSample Source type']), flags=re.IGNORECASE):
        return True

    non_human_sources = r'(animal|environmental|food)'
    if re.search(non_human_sources, str(row['Assembly BioSample Source type']), flags=re.IGNORECASE):
        return False

    if re.search(search_list, str(row['Assembly BioSample Isolation source']), flags=re.IGNORECASE):
        return True
    
    if re.search(search_list, str(row['Assembly BioProject Lineage Title']), flags=re.IGNORECASE):
        return True

    if re.search(search_list, str(row['Assembly BioSample Models']), flags=re.IGNORECASE):
        return True
    # for val in row:
    #     if re.search(search_list, str(val), flags=re.IGNORECASE):
    #         return True
        
    return False

ncbi_info['is_human'] = ncbi_info.apply(is_human, axis=1)

In [15]:
ncbi_info['is_human'].value_counts()

is_human
False    183762
True     113602
Name: count, dtype: int64

### Manually categorize most prevalent isolate sources

In [16]:
human_derived = ncbi_info[ncbi_info['is_human']==True]
human_derived.drop_duplicates('Assembly Accession')

,Assembly Accession,Assembly BioProject Lineage Title,Assembly BioSample BioProject Title,Assembly BioSample Project name,Assembly BioSample Description Title,Assembly BioSample Breed,Assembly BioSample Host,Assembly BioSample Ecotype,Assembly BioSample Cultivar,Assembly BioSample Host disease,Assembly BioSample Isolation source,Assembly BioSample Source type,Assembly BioSample Tissue,Assembly BioSample Models,is_human
0,GCA_000740545.1,Escherichia coli strain:CS01 Genome sequencing...,NaN,NaN,Microbe sample from Escherichia coli,NaN,homo sapiens,NaN,NaN,NaN,human fecal sample,NaN,NaN,"Microbe, viral or environmental",True
1,GCA_000740625.1,Escherichia coli strain:CS03 Genome sequencing...,NaN,NaN,Microbe sample from Escherichia coli,NaN,homo sapiens,NaN,NaN,NaN,human fecal sample,NaN,NaN,"Microbe, viral or environmental",True
3,GCA_000936245.1,Comparative genomics to delineate pathogenic p...,NaN,NaN,STEC strain from patient with HUS,NaN,homo sapiens,NaN,NaN,NaN,human faeces,NaN,NaN,Generic,True
4,GCA_000936475.1,Comparative genomics to delineate pathogenic p...,NaN,NaN,Non-O157 STEC,NaN,homo sapiens,NaN,NaN,NaN,human faeces,NaN,NaN,Generic,True
5,GCA_000937275.1,Comparative genomics to delineate pathogenic p...,NaN,NaN,STEC strain from patient with HUS,NaN,homo sapiens,NaN,NaN,NaN,human faeces,NaN,NaN,Generic,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306262,GCA_900760205.1,Prokaryotic genomes for 2058 new species assem...,NaN,Global Human Gut MAGs,Human gut MAG #3 derived from sample SRS013687,NaN,NaN,NaN,NaN,NaN,human stool,NaN,NaN,Generic,True
306263,GCA_900766785.1,Prokaryotic genomes for 2058 new species assem...,NaN,Global Human Gut MAGs,Human gut MAG #104 derived from sample SRS475647,NaN,NaN,NaN,NaN,NaN,human stool,NaN,NaN,Generic,True
306272,GCA_900551745.1,A new genomic blueprint of the human gut micro...,NaN,NaN,Metagenome-Assembled Genome (MAG),NaN,NaN,NaN,NaN,NaN,human gut,NaN,NaN,Generic,True
306273,GCA_900539805.1,A new genomic blueprint of the human gut micro...,NaN,NaN,Metagenome-Assembled Genome (MAG),NaN,NaN,NaN,NaN,NaN,human gut,NaN,NaN,Generic,True


In [17]:
human_derived['Assembly BioSample Isolation source'] = human_derived['Assembly BioSample Isolation source'].str.strip(' ')
human_derived['Assembly BioSample Isolation source'].value_counts(dropna=False).head(20)

/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_28380/3188728986.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  human_derived['Assembly BioSample Isolation source'] = human_derived['Assembly BioSample Isolation source'].str.strip(' ')


Assembly BioSample Isolation source
NaN                   15801
blood                 11120
missing               10284
feces                  7093
sputum                 5025
not known              4674
stool                  4414
nasopharynx            4253
urine                  3984
human gut              3085
patient                1722
faecal                 1606
unknown                1567
human stool            1461
csf                    1189
respiratory system     1101
nares                  1080
rectal swab             895
not collected           873
human                   838
Name: count, dtype: int64

In [18]:
top_source_mappings = {                   
'blood':'blood',
'feces':'gastrointestinal',
'sputum': 'respiratory',
'stool':'gastrointestinal',
'nasopharynx': 'respiratory',           
'urine':'urine',
'gut': 'gastrointestinal',
'faecal':'gastrointestinal',
'csf':'other',
'respiratory system': 'respiratory',
'nares':'respiratory',                  
'rectal swab': 'gastrointestinal',
'vagina':'female reproductive tract',
'cervix':'female reproductive tract',
'uterus':'female reproductive tract',
'skin':'skin',
'nasophaynx':'respiratory',
'throat':'respiratory',
'wound': 'other',
'derived from human gut metagenome': 'gastrointestinal',
'blood culture': 'blood',
'vaginal/rectal': 'unknown',
'dental plaque': 'oral',
'gastrointestinal tract': 'gastrointestinal',
'lower respiratory tract secrection': 'gastrointestinal',
'oral cavity': 'oral',
'cerebrospinal fluid': 'other',
'urinary tract': 'urine',
'wound secretion': 'other',
'skin, soft tissue infection': 'skin',
'fecal sample': 'gastrointestinal',
'gastric biopsy': 'gastrointestinal',
'excreted bodily substance': 'unknown',
'fecal material': 'gastrointestinal',
'respiratory': 'respiratory',
'stool sample': 'gastrointestinal',
'tracheal aspirate': 'respiratory',
'skin sore / abscess / burns / iv site': 'skin',
'throat swab': 'oral',
'catheter': 'urine',
'pus': 'other',
'bronchoalveolar lavage': 'respiratory',
'stomach': 'gastrointestinal',
'wastewater influent sample': 'non-human',
'lung': 'respiratory',
'bal': 'respiratory',
'wound swab': 'other',
'freshwater sample from downstream of wastewater treatment plant': 'non-human',
'cervical': 'female reproductive tract',
'urinogenital fluid': 'unknown',
'oral cavity- mouth': 'oral',
'subgingival dental plaque': 'oral',
'pleural fluid': 'respiratory',
'rectal': 'gastrointestinal',
'wastewater effluent sample': 'non-human',
'endotracheal aspirate': 'respiratory',
'tracheal secretion': 'respiratory',
'nasal': 'respiratory',
'eye': 'other',
'perirectal': 'gastrointestinal',
'bronchial washing': 'respiratory',
'saliva': 'oral',
'nasal swab': 'female reproductive tract',
'vaginal': 'female reproductive tract',
'human gut stool': 'gastrointestinal',
'puncture fluids': 'other',
'portion of sputum': 'respiratory',
'bone': 'other',
'abdominal fluid': 'gastrointestinal',
'ear': 'other',
'urine cvs': 'urine',
'peritoneal fluid': 'other',
'rectum': 'gastrointestinal',
'rectal swap': 'gastrointestinal',
'genital': 'unknown',
'ucc isolate': 'urine',
'gut microbiota': 'gastrointestinal',
'vaginal microbiome': 'female reproductive tract',
'nose': 'respiratory',
'bile': 'gastrointestinal',
'host stool sample': 'gastrointestinal',
'derived from human gut metagenome': 'gastrointestinal',
'human gut stool': 'gastrointestinal',
'peripheral blood': 'blood',
'freshwater sample from upstream of wastewater treatment plant': 'non-human',
'pooled cattle faecal samples collected from floor of farm': 'non-human',
'pooled pig faecal samples collected from floor of farm': 'non-human',
'pooled sheep faecal samples collected from floor of farm': 'non-human',
'oral': 'oral',
'bloodstream': 'blood',
'vaginal swab': 'female reproductive tract',
'trach aspirate': 'respiratory',
'gut stool': 'gastrointestinal',
'faeces':'gastrointestinal',
'household surface':'non-human',
'derived from  gut metagenome':'gastrointestinal',
'wound swab sample': 'other',
'osteomyelitis': 'other',
'infant feces': 'gastrointestinal',
'bronchial alveolar lavage': 'respiratory',
'stool, clinical': 'gastrointestinal',
'stool sample of individual with bacteremia': 'gastrointestinal',
'soft-skin infection or carrier': 'skin',
'respiratory tract': 'respiratory',
'bronchial washings': 'respiratory',
'surgical wound': 'other',
'subgingival plaque': 'oral',
'pharygitis': 'respiratory',
'trachael aspirate': 'respiratory',
'intestine': 'gastrointestinal',
'lungs': 'respiratory',
'(feces)': 'gastrointestinal',
'urine sample': 'urine',
'skin swab': 'skin',
'aspirate': 'respiratory',
'water': 'non-human',
'environmental': 'non-human',
'synovial fluid': 'other',
'sputum sample': 'respiratory',
'clinical specimen, blood': 'blood',
'ascites': 'other',
'korean adult feces': 'gastrointestinal',
'wound or pus swab': 'other',
'conjunctiva': 'other',
'wound abdominal': 'other',
'endotracheal tube': 'respiratory',
'oropharynx': 'respiratory',
'brain abscess': 'other',
'urethra': 'urine',
'nasopharyngeal swab': 'respiratory',
'joint': 'other',
'ascitic fluid': 'other',
'sputum or oropharyngeal':'respiratory',
'vaginal-rectal':'unknown',
'pulmonary':'respiratory',
'brain':'other',
'sputamentum':'respiratory',
'milk':'other',
'gastric tissue':'gastrointestinal',
'pharynx':'respiratory',
}


unspecified = ['', 'not known', 'patient', 'unknown', 'not collected', 'human','missing',
       'clinical sample', 'bodily fluid', 'not available', 'nottingham','not provided', 'clinical', 'not applicable', 'clinical isolate',
       'sterile site', 'hospital', 'biopsy','clinic', 'tissue','clinical specimen', 'not available: to be reported later','clinical specimen',
       'csf or blood culture',
'blood cerebrospinal fluid']

for f in unspecified:
    top_source_mappings[f] = 'unknown'

    

In [19]:
human_derived['Assembly BioSample Isolation source'] = human_derived['Assembly BioSample Isolation source'].fillna(human_derived['Assembly BioSample Tissue'])
human_derived['Assembly BioSample Isolation source'] = human_derived['Assembly BioSample Isolation source'].fillna('')

human_derived['human_subcategory'] = human_derived['Assembly BioSample Isolation source'].str.replace('human','').str.strip(' ').map(top_source_mappings)
human_derived['human_subcategory'].value_counts(dropna=False)

/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_28380/2060224959.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  human_derived['Assembly BioSample Isolation source'] = human_derived['Assembly BioSample Isolation source'].fillna(human_derived['Assembly BioSample Tissue'])
/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_28380/2060224959.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  human_derived['Assembly BioSample Isolation source'] = human_derived['Assembly BioSample Isolatio

human_subcategory
unknown                      42133
gastrointestinal             22546
respiratory                  15505
blood                        11798
NaN                          10146
urine                         4776
other                         3609
oral                           994
skin                           981
female reproductive tract      696
non-human                      418
Name: count, dtype: int64

In [20]:
model_to_site = {

'MIGS.ba,MIGS/MIMS/MIMARKS.human-gut': 'gastrointestinal',
'MIGS.ba.human-gut': 'gastrointestinal',
'MIGS.ba.human-skin': 'skin',
'MIGS.ba,MIGS/MIMS/MIMARKS.human-oral': 'oral',
'MIGS.ba,MIGS/MIMS/MIMARKS.human-skin': 'skin',
'MIGS.ba,MIGS/MIMS/MIMARKS.human-vaginal': 'female reproductive tract',
'MIGS.ba.human-vaginal': 'female reproductive tract',
'MIGS.ba.human-oral': 'oral',
'MIMS.me,MIGS/MIMS/MIMARKS.human-gut': 'gastrointestinal',
'MIMAG.human-vaginal': 'female reproductive tract',
'MIMAG.human-gut': 'gastrointestinal',
'MIMARKS.survey,MIGS/MIMS/MIMARKS.human-gut': 'gastrointestinal',
'MIGS.ba.human-gut.4.0': 'gastrointestinal',
'MIGS.ba,MIGS/MIMS/MIMARKS.plant-associated': 'non-human',
'MIGS.ba,MIGS/MIMS/MIMARKS.soil': 'non-human',
'MIGS.ba,MIGS/MIMS/MIMARKS.water': 'non-human',
'Pathogen,environmental/food/other': 'non-human',
}


def fill_subcategory_with_model(row):

    if  ~pd.isna(row['human_subcategory']) and row['human_subcategory'] != 'unknown':
        return row['human_subcategory']

    if row['Assembly BioSample Models'] in model_to_site:
        return model_to_site[row['Assembly BioSample Models']]

    else:
        return row['human_subcategory']

    



In [21]:
human_derived['human_subcategory'] = human_derived.apply(fill_subcategory_with_model, axis=1)

/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_28380/3335337083.py:25: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  if  ~pd.isna(row['human_subcategory']) and row['human_subcategory'] != 'unknown':
/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_28380/3463744791.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  human_derived['human_subcategory'] = human_derived.apply(fill_subcategory_with_model, axis=1)


In [22]:
human_derived['human_subcategory'].value_counts(dropna=False)

human_subcategory
unknown                      39078
gastrointestinal             25062
respiratory                  15505
blood                        11798
NaN                          10146
urine                         4776
other                         3609
skin                          1349
oral                          1048
female reproductive tract      807
non-human                      424
Name: count, dtype: int64

There are 1212 samples with Assembly BioProject Lineage Title = Bacterial isolates from intensive care units. These do not have any metadata indicating body sites, so I will classify all of these as unknown.

In [23]:
human_derived[
    (human_derived['human_subcategory'].isna())
    & (human_derived['Assembly BioProject Lineage Title'] == 'Bacterial isolates from intensive care units')]

,Assembly Accession,Assembly BioProject Lineage Title,Assembly BioSample BioProject Title,Assembly BioSample Project name,Assembly BioSample Description Title,Assembly BioSample Breed,Assembly BioSample Host,Assembly BioSample Ecotype,Assembly BioSample Cultivar,Assembly BioSample Host disease,Assembly BioSample Isolation source,Assembly BioSample Source type,Assembly BioSample Tissue,Assembly BioSample Models,is_human,human_subcategory


In [24]:
human_derived.loc[
    human_derived['human_subcategory'].isna()
    & (human_derived['Assembly BioProject Lineage Title'] == 'Bacterial isolates from intensive care units'),
    'human_subcategory'
] = 'unknown'

All of the urinary microbiome genomes are from urine

In [25]:
human_derived[
    ((human_derived['human_subcategory'].isna()) | (human_derived['human_subcategory']=='unknown')) &
     (human_derived['Assembly BioProject Lineage Title'] == 'Female Urinary Microbiota Genome Sequencing')]

human_derived.loc[
    ((human_derived['human_subcategory'].isna()) | (human_derived['human_subcategory']=='unknown'))
    & (human_derived['Assembly BioProject Lineage Title'] == 'Female Urinary Microbiota Genome Sequencing'),
    'human_subcategory'
] = 'urine'

In [26]:
human_unknown = human_derived[human_derived['human_subcategory'].isna()]
human_unknown.to_csv('unclassified_human_samples.csv')

human_known = human_derived[~human_derived['human_subcategory'].isna()]
human_known.to_csv('manually_classified_human_samples.csv')

human_known.drop_duplicates('Assembly Accession').shape, human_unknown.drop_duplicates('Assembly Accession').shape

((89483, 16), (9200, 16))

In [27]:
unspecified = ['', 'not known', 'patient', 'unknown', 'not collected', 'human','missing',
       'clinical sample', 'bodily fluid', 'not available', 'nottingham','not provided', 'clinical', 'not applicable', 'clinical isolate',
       'sterile site', 'hospital', 'biopsy','clinic', 'tissue','clinical specimen', 'not available: to be reported later','clinical specimen',
       'csf or blood culture',
'blood cerebrospinal fluid']

unclassified_isolate_sources =  human_unknown[~human_unknown['Assembly BioSample Isolation source'].isin(unspecified)]['Assembly BioSample Isolation source'].value_counts().index


In [31]:
prompt = """You are a data encoder for biological data. You will be provided the free-text isolation source of a specimen.

From which body site (source) was this specimen obtained? Pick from the following list of body sites:

- gastrointestinal
- female reproductive tract
- urine
- skin
- respiratory
- oral
- blood 
- other
- unknown
- non-human

Your response should be only one of these as a single word. The response must be verbatim one of the members of the list.
Do not infer only based on mention of a disease; there must be specific reference to the body site the specimen was collected from. 

Here are some examples of inputs and correct outputs:

Input:
thracheal aspirate
Output:
respiratory

Input:
finger scar
Output:
skin

Input:
genital smear
Output:
unknown

Input:
stool sample from patient (carrier)
Output:
gastrointestinal

Input:
feces, human
Output:
gastrointestinal

Input:
pooled pig fecal samples collected from floor of farm
Output:
non-human

Input:
blood sample from a 45 year old woman who was hospitalized in icu
Output:
blood

Input:
breast milk
Output:
other

Input:
wound right thigh
Output:
skin

Input:
lymphnode
Output:
other

Input:
blood culture from newborn
Output:
blood

Input:
scleral scraping of eye
Output:
other

Input:
pus leg
Output:
skin

Input:
urine, suprapubic aspiration
Output:
urine

Input:
human-outbreak
Output:
unknown

Input:
oral cavity, subgingival dental plaque
Output:
oral

Input:
middle ear
Output:
other

Input:
vesical catheter
Output:
urine

Input:
spontaneous mutant of kp1099 selected on meropenem/vaborbactam, both at 8 mg/l
Output:
unknown

Input:
bood
Output:
blood

Input:
labial angle of a patient with cheilitis angularis
Output:
oral

Input:
vaginal swab sample
Output:
female reproductive tract

Input:
orthopedic infection
Output:
other

Input:
abscess
Output:
unknown

Input:
amniotic fluid
Output:
female reproductive tract
"""


In [32]:
len(unclassified_isolate_sources)

3028

In [ ]:
model = "google/gemma-2-9b-it"
provider = "nebius"

    
client = InferenceClient(
    provider=provider,
    api_key=os.environ["HF_TOKEN"],
)

outfile = f'{model.split('/')[1]}_classifications.csv'
!rm {outfile}

start_time = time.time()
processed_count = 0
for source in unclassified_isolate_sources:


    sample_info = f'\nHere is the sample for you to classify:\nInput:\n{source}\nOutput:'

    while True:
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": prompt+sample_info,
                    }
                ],
                )
            break
        except HfHubHTTPError as e:
            print(f"HfHubHTTPError: {e}. Sleeping 30s then retrying...")
            time.sleep(30)

    
    classification = completion.choices[0].message['content'].strip('\n').strip(' ')

    with open(outfile, 'a') as f:
        f.write(f'{source}|{classification}\n')

    processed_count += 1

    print([source, classification])

    if processed_count % 100 == 0:

        elapsed_time = time.time() - start_time
        avg_time_per_sample = elapsed_time / processed_count
        remaining_samples = total_lines - processed_count
        estimated_remaining_time = remaining_samples * avg_time_per_sample

        print(f"Progress: {processed_count}/{total_lines} processed")
        print(f"Average time per sample: {avg_time_per_sample:.2f} seconds")
        print(f"Estimated time remaining: {estimated_remaining_time/3600:.1f} hours")
        print(f"Estimated completion: {time.ctime(time.time() + estimated_remaining_time)}")
        print("-" * 50)


elapsed_time = time.time() - start_time
print(model)
print(f"Total time: {elapsed_time} seconds")
print(f"Average time per sample: {elapsed_time/total_run:.2f} seconds")
print()

rm: gemma-2-9b-it_classifications.csv: No such file or directory
['patient with urinary tract infection', 'urine']
['soft tissue', 'skin']
['commercial dietary supplements', 'other']
['cystic fibrosis patient', 'unknown']
['blood from patient with bacteremia', 'blood']
['abscess', 'unknown']
['culture', 'unknown']
['healthy adult 3c', 'unknown']
['groin', 'skin']
['farm workers', 'unknown']
['clinical sample from patient with gastroenteritis', 'gastrointestinal']
['infection', 'unknown']
['cerebro-spinal fluid', 'other']
['patient sample', 'unknown']
['lavage fluid', 'unknown']
['urethral swab', 'urine']
['laboratory strain', 'unknown']
['middle ear', 'other']
['trachea', 'respiratory']
['rectal swab from healthy college student', 'gastrointestinal']
['lesion swab', 'skin']
['body fluid', 'unknown']
['gastric tissue biopsy', 'gastrointestinal']
['patient fluids', 'unknown']
['bronchoalveolar lavage fluid', 'respiratory']
['wound/abscess', 'skin']
['perineum swab', 'female reproductive 